In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# --- STEP 1: HIGH-PERFORMANCE INSTALLATION ---
# On H100, we skip xformers and rely on native Flash Attention 2
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

import json
import torch
import pandas as pd
from unsloth import FastLanguageModel

# Suppress the factory registration warnings if they are distracting
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

print("Setup Complete. Using native Flash Attention on:", torch.cuda.get_device_name(0))

In [ ]:
# --- REPAIR BLOCK ---
# 1. Uninstall existing versions to avoid conflicts
!pip uninstall -y unsloth unsloth_zoo

# 2. Reinstall both directly from GitHub to ensure they are the absolute latest and synced
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

print("Reinstallation complete. Please RESTART YOUR SESSION now.")

In [ ]:
import json
import torch
import pandas as pd
from unsloth import FastLanguageModel

# Suppress the factory registration warnings if they are distracting
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

print("Setup Complete. Using native Flash Attention on:", torch.cuda.get_device_name(0))

In [ ]:
import re
import random
from tqdm.notebook import tqdm
from difflib import SequenceMatcher

# Helper for JSON extraction (Fixes the UNKNOWN intent issue)
def extract_json(raw_output):
    """Robustly extracts JSON and ensures required keys exist."""
    try:
        # Regex to find anything between curly braces
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            # Basic cleanup: replace single quotes with double quotes
            clean_json = json_match.group(0).replace("'", '"')
            data = json.loads(clean_json)
            
            # Ensure both keys exist; use defaults if missing
            return {
                "text": data.get("text", raw_output),
                "intent": data.get("intent", "UNKNOWN_INTENT")
            }
    except Exception:
        pass
        
    # Final fallback if regex or JSON parsing fails entirely
    return {"text": raw_output, "intent": "UNKNOWN_PARSING_ERROR"}
    
# Helper to check for repetition (The Critic)
def is_repetitive(text, history_list, threshold=0.85):
    for prev_text in history_list:
        if SequenceMatcher(None, text, prev_text).ratio() > threshold:
            return True
    return False

print("Setup Complete.")

In [ ]:
# # --- STEP 2: MODEL INITIALIZATION ---
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
#     max_seq_length = 2048,
#     load_in_4bit = True,
# )

# # Unsloth will handle the patching for H100 automatically
# FastLanguageModel.for_inference(model)
# print("Model ready for high-speed Banglish generation.")

In [ ]:
# # --- STEP 2: UPGRADED MODEL INITIALIZATION ---
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit", # UPGRADED to 32B
#     max_seq_length = 4096, # Increased context window
#     load_in_4bit = True,
# )

# FastLanguageModel.for_inference(model)
# print("32B Parameter Model Ready.")

In [ ]:
# --- STEP 2: UPGRADED 72B MODEL INITIALIZATION ---
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    # The 72B model provides the highest level of reasoning for Banglish
    model_name = "unsloth/Qwen2.5-72B-Instruct-bnb-4bit",
    max_seq_length = 4096, 
    load_in_4bit = True,
)

# Optimized for H100
FastLanguageModel.for_inference(model)
print("72B Model Loaded. VRAM used: ~45GB.")

In [ ]:
# # --- STEP 3: PIPELINE LOGIC ---
# class GenerationPipeline:
#     def __init__(self, archetypes):
#         self.archetypes = archetypes

#     def generate_turn(self, role, history, context, behavior):
#         # Extract the language style from the context string
#         if "Pure Bangla" in context:
#             lang_instruction = "Pure Bangla (no English words at all)."
#         elif "Pure English" in context:
#             lang_instruction = "Pure English (no Bangla words at all)."
#         else:
#             lang_instruction = "Banglish (a natural mix of Bangla and English as used in BD social media)."

#         system_msg = f"You are a {role} in a Bangladeshi marketplace. Style: {lang_instruction}. "
#         if role == "Customer":
#             system_msg += f"Behavior: {behavior}"
#         else:
#             system_msg += "Behavior: Professional seller, helpful but firm on price."

#         prompt = f"Product Context: {context}\nHistory: {history}\nNext turn ({role}):"
        
#         messages = [
#             {"role": "system", "content": f"{system_msg}\nOutput JSON format: {{'text': '...', 'intent': '...'}}"},
#             {"role": "user", "content": prompt}
#         ]
        
#         inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
#         attention_mask = (inputs != tokenizer.pad_token_id).long().to("cuda")
        
#         outputs = model.generate(
#             input_ids=inputs, 
#             attention_mask=attention_mask,
#             max_new_tokens=200, 
#             temperature=0.85,
#             pad_token_id=tokenizer.eos_token_id
#         )
        
#         raw_out = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
#         try:
#             return json.loads(raw_out)
#         except:
#             return {"text": raw_out, "intent": "UNKNOWN"}

#     def run_simulation(self, product, behavior_type, turns=6):
#         behavior = self.archetypes[behavior_type]
#         history_str = ""
#         dialogue = []
        
#         for i in range(turns):
#             role = "Customer" if i % 2 == 0 else "Merchant"
#             turn = self.generate_turn(role, history_str, product, behavior)
            
#             dialogue.append({
#                 "role": role,
#                 "text": turn["text"],
#                 "intent": turn["intent"],
#                 "archetype": behavior_type if role == "Customer" else "N/A"
#             })
#             history_str += f"{role}: {turn['text']} \n"
            
#         return dialogue

In [ ]:
# --- STEP 3: PIPELINE LOGIC WITH DST ---
class GenerationPipeline:
    def __init__(self, archetypes):
        self.archetypes = archetypes

    def generate_turn(self, role, history, context, behavior, fulfilled_goals):
        # Language instruction logic
        if "Pure Bangla" in context:
            lang_inst = "Pure Bangla (No English)."
        elif "Pure English" in context:
            lang_inst = "Pure English."
        else:
            lang_inst = "Banglish (Natural mix of Bangla/English)."

        system_msg = f"You are a {role} in a Bangladeshi marketplace. Style: {lang_inst}. "
        if role == "Customer":
            system_msg += f"Behavior: {behavior}. "
        
        # State Tracking Instruction (Dialogue State Tracking)
        state_msg = f"Already discussed/done: {', '.join(fulfilled_goals) if fulfilled_goals else 'Nothing yet'}. "
        system_msg += f"{state_msg} DO NOT repeat questions or information already provided."

        prompt = f"Product Context: {context}\nHistory:\n{history}\nNext turn ({role}) as JSON:"
        
        messages = [
            {"role": "system", "content": f"{system_msg}\nFormat: {{'text': '...', 'intent': '...'}}"},
            {"role": "user", "content": prompt}
        ]
        
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        
        # Explicitly create attention mask for stability
        attention_mask = (inputs != tokenizer.pad_token_id).long().to("cuda")
        
        # FIXED: Changed presence_penalty to repetition_penalty
        outputs = model.generate(
            input_ids=inputs, 
            attention_mask=attention_mask,
            max_new_tokens=150, 
            temperature=0.6,
            repetition_penalty=1.2, # Values > 1.0 discourage repetition
            pad_token_id=tokenizer.eos_token_id
        )
        
        raw_out = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
        return extract_json(raw_out)

    def run_simulation(self, product_context, behavior_type, turns=6):
        behavior = self.archetypes[behavior_type]
        history_str = ""
        history_list = {"Customer": [], "Merchant": []}
        fulfilled_goals = []
        dialogue = []
        
        for i in range(turns):
            role = "Customer" if i % 2 == 0 else "Merchant"
            turn_data = self.generate_turn(role, history_str, product_context, behavior, fulfilled_goals)
            
            # The extract_json helper now guarantees 'text' and 'intent' keys exist
            text = turn_data.get("text", "Error: No text generated")
            intent = turn_data.get("intent", "UNKNOWN")

            # Critic Check: If the turn is repetitive, try one more time
            if is_repetitive(text, history_list[role]):
                 # Nudge the model to move forward
                 turn_data = self.generate_turn(role, history_str + " (SYSTEM: Avoid repeating previous points.)", product_context, behavior, fulfilled_goals)
                 text = turn_data.get("text", text)
                 intent = turn_data.get("intent", intent)

            dialogue.append({
                "role": role,
                "text": text,
                "intent": intent
            })
            
            fulfilled_goals.append(intent)
            history_list[role].append(text)
            history_str += f"{role}: {text}\n"
            
        return dialogue

In [ ]:
# # --- STEP 4: BATCH EXECUTION ---
# ARCHETYPES = {
#     "Skeptic": "You doubt the quality. Ask for proof and real photos in Banglish.",
#     "LowBaller": "You want a massive discount. Use slang like 'Mama, budget kom'.",
#     "Urgent": "You need it now. Ask for bKash details immediately.",
#     "Ghoster": "Ask many questions, then end with 'Acha bhabte hobe'."
# }

# pipeline = GenerationPipeline(ARCHETYPES)
# dataset = []

# # Scaling up for the H100
# for i in range(25): 
#     for a_type in ARCHETYPES.keys():
#         chat = pipeline.run_simulation("Product: Premium Panjabi, Price: 3500 BDT", a_type)
#         dataset.append({"id": f"chat_{len(dataset)}", "dialogue": chat})

# df = pd.DataFrame(dataset)
# df.to_json("buyer_seller_dataset_h100.json", orient="records", indent=4)
# print(f"Generation Complete: {len(dataset)} chats produced.")

In [ ]:
# import random
# from tqdm.notebook import tqdm
# import pandas as pd

# # 1. UPDATED ARCHETYPES
# ARCHETYPES = {
#     "Skeptic": "You doubt the quality. Ask for proof and real photos.",
#     "LowBaller": "You want a massive discount. Use local bargaining tactics like 'Mama, budget kom'.",
#     "Urgent": "You need it immediately. Ask for bKash details fast.",
#     "Ghoster": "Ask many questions, then end with 'Acha bhabte hobe' (Need to think).",
#     "Comparator": "Tell the seller you saw this cheaper elsewhere and ask for a price match.",
#     "DeliveryObsessed": "You only care about how fast it arrives. Demand delivery by tomorrow morning.",
#     "TrustSeeker": "You are worried about scams. Ask for 'Cash on Delivery' and a physical address.",
#     "BulkBuyer": "Ask if the price reduces if you buy 5 or 10 pieces at once."
# }

# # 2. LANGUAGE OPTIONS
# LANG_STYLES = ["Pure Bangla", "Pure English", "Banglish"]

# # 3. EXPANDED PRODUCT DB (Adding more variety)
# PRODUCT_DB = [
#     {"product": "Smart Watch (Series 8)", "price": "4500 BDT", "category": "Electronics"},
#     {"product": "Jamdani Saree", "price": "12000 BDT", "category": "Clothing"},
#     {"product": "Walton Blender", "price": "3200 BDT", "category": "Home Appliance"},
#     {"product": "iPhone 15 Pro (Used)", "price": "85000 BDT", "category": "Smartphone"},
#     {"product": "Gaming Mouse (Logitech)", "price": "1800 BDT", "category": "Peripherals"},
#     {"product": "Organic Honey (Sundarbans)", "price": "950 BDT", "category": "Food"}
# ]

# pipeline = GenerationPipeline(ARCHETYPES)
# dataset = []

# # Total = 120 chats (15 iterations * 8 archetypes randomly matched)
# ITERATIONS = 15 
# total_steps = ITERATIONS * len(PRODUCT_DB)

# with tqdm(total=total_steps, desc="Generating Triple-Language Dataset") as pbar:
#     for i in range(ITERATIONS):
#         for product_info in PRODUCT_DB:
#             # Randomize language and behavior
#             lang_style = random.choice(LANG_STYLES)
#             a_type = random.choice(list(ARCHETYPES.keys()))
            
#             context = f"{product_info['product']}, Price: {product_info['price']}. Language Style: {lang_style}."
            
#             try:
#                 chat = pipeline.run_simulation(context, a_type)
#                 dataset.append({
#                     "id": f"chat_{len(dataset)}",
#                     "product": product_info['product'],
#                     "category": product_info['category'],
#                     "language": lang_style,
#                     "archetype": a_type,
#                     "dialogue": chat
#                 })
#             except Exception as e:
#                 pass # Silently skip errors to keep the bar moving
            
#             pbar.update(1)

# # SAVE
# df = pd.DataFrame(dataset)
# df.to_json("comprehensive_market_dataset_v2.json", orient="records", indent=4)
# print(f"Success! {len(df)} chats generated.")

In [ ]:
import json

# 1. Load your existing "broken" file
input_filename = "comprehensive_market_dataset_v2.json" 
output_filename = "readable_bangla_dataset.json"

with open(input_filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Save it again, but force it to keep the native characters
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Success! Open '{output_filename}' in VS Code to see the Bengali text.")

In [ ]:
# # --- STEP 4: ENRICHED EXECUTION ---
# PRODUCT_DB = [
#     {"product": "Aarong Cotton Panjabi", "price": "3500 BDT", "category": "Clothing"},
#     {"product": "Organic Honey (Sundarbans)", "price": "950 BDT", "category": "Food"},
#     {"product": "Leather Laptop Bag", "price": "2800 BDT", "category": "Accessories"},
#     {"product": "Xiaomi Air Purifier", "price": "15500 BDT", "category": "Electronics"},
#     {"product": "Non-stick Cookware Set", "price": "4200 BDT", "category": "Kitchen"},
#     {"product": "Induction Cooktop", "price": "5500 BDT", "category": "Home Appliance"},
#     {"product": "Sony Wireless Headphones", "price": "12500 BDT", "category": "Electronics"}
# ]

# ARCHETYPES = {
#     "Skeptic": "You doubt the quality. Ask for proof and real photos.",
#     "LowBaller": "You want a massive discount. Use local bargaining tactics like 'Mama, budget kom'.",
#     "Urgent": "You need it immediately. Ask for bKash details fast.",
#     "Ghoster": "Ask many questions, then end with 'Acha bhabte hobe' (Need to think).",
#     "Comparator": "Tell the seller you saw this cheaper elsewhere and ask for a price match.",
#     "DeliveryObsessed": "You only care about how fast it arrives. Demand delivery by tomorrow morning.",
#     "TrustSeeker": "You are worried about scams. Ask for 'Cash on Delivery' and a physical address.",
#     "BulkBuyer": "Ask if the price reduces if you buy 5 or 10 pieces at once."
# }

# LANGUAGES = ["Pure Bangla", "Pure English", "Banglish"]
# pipeline = GenerationPipeline(ARCHETYPES)
# dataset = []

# # Generate 60 diverse chats
# for i in tqdm(range(5), desc="Batches"):
#     for prod in PRODUCT_DB:
#         for lang in LANGUAGES:
#             a_type = random.choice(list(ARCHETYPES.keys()))
#             context = f"{prod['product']}, Price: {prod['price']}. Language: {lang}."
            
#             chat = pipeline.run_simulation(context, a_type)
#             dataset.append({
#                 "product": prod['product'],
#                 "language": lang,
#                 "archetype": a_type,
#                 "dialogue": chat
#             })

# # Final Save
# pd.DataFrame(dataset).to_json(
#     "refined_dataset.json", 
#     orient="records", 
#     indent=4, 
#     force_ascii=False
# )

# print(f"Generated {len(dataset)} high-quality, non-repetitive chats.")

In [ ]:
# --- STEP 4: ENRICHED EXECUTION WITH NEW PRODUCT CATEGORIES ---

PRODUCT_DB = [
    # High-Value Electronics (Testing Trust & Skepticism)
    {"product": "iPhone 15 Pro Max (Used)", "price": "115,000 BDT", "category": "Smartphone"},
    {"product": "Sony PS5 Console", "price": "58,000 BDT", "category": "Gaming"},
    
    # Home & Kitchen (Testing Delivery & Bulk needs)
    {"product": "LG Inverter Refrigerator", "price": "65,000 BDT", "category": "Home Appliance"},
    {"product": "Miyako Electric Kettle", "price": "1,500 BDT", "category": "Kitchen"},
    
    # Traditional & Luxury (Testing Banglish & Nuance)
    {"product": "Authentic Rajshahi Silk Saree", "price": "8,500 BDT", "category": "Clothing"},
    {"product": "Premium Katmon Ghee (1kg)", "price": "1,800 BDT", "category": "Food"},
    
    # Health & Lifestyle (Testing Comparative & Ghoster behavior)
    {"product": "Adjustable Dumbbell Set", "price": "4,200 BDT", "category": "Fitness"},
    {"product": "CeraVe Moisturizing Cream", "price": "2,400 BDT", "category": "Skincare"}
]

# (The rest of your ARCHETYPES, LANGUAGES, and Tracker logic remains the same)
ARCHETYPES = {
    "Skeptic": "You doubt the quality. Ask for proof and real photos.",
    "LowBaller": "You want a massive discount. Use local bargaining tactics like 'Mama, budget kom'.",
    "Urgent": "You need it immediately. Ask for bKash details fast.",
    "Ghoster": "Ask many questions, then end with 'Acha bhabte hobe' (Need to think).",
    "Comparator": "Tell the seller you saw this cheaper elsewhere and ask for a price match.",
    "DeliveryObsessed": "You only care about how fast it arrives. Demand delivery by tomorrow morning.",
    "TrustSeeker": "You are worried about scams. Ask for 'Cash on Delivery' and a physical address.",
    "BulkBuyer": "Ask if the price reduces if you buy 5 or 10 pieces at once."
}

LANGUAGES = ["Pure Bangla", "Pure English", "Banglish"]
pipeline = GenerationPipeline(ARCHETYPES)
dataset = []

# Progress Tracker logic
num_batches = 5
total_chats_per_batch = len(PRODUCT_DB) * len(LANGUAGES)
master_pbar = tqdm(range(num_batches), desc="Master Progress (Batches)")

for i in master_pbar:
    sub_pbar = tqdm(total=total_chats_per_batch, desc=f"Batch {i+1} Details", leave=False)
    for prod in PRODUCT_DB:
        for lang in LANGUAGES:
            a_type = random.choice(list(ARCHETYPES.keys()))
            context = f"{prod['product']}, Price: {prod['price']}. Language: {lang}."
            sub_pbar.set_postfix({"Product": prod['product'][:12], "Type": a_type})
            
            try:
                chat = pipeline.run_simulation(context, a_type)
                dataset.append({
                    "batch": i + 1,
                    "product": prod['product'],
                    "category": prod['category'],
                    "language": lang,
                    "archetype": a_type,
                    "dialogue": chat
                })
            except Exception as e:
                print(f"\nError processing {prod['product']}: {e}")
            
            sub_pbar.update(1)
    sub_pbar.close()

# Final Save
pd.DataFrame(dataset).to_json("refined_dataset_72B_V2.json", orient="records", indent=4, force_ascii=False)
print(f"\n✅ SUCCESS: Generated {len(dataset)} chats with the new product mix.")